> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t03_rerun_raw_to_bronze` · notebook `01_raw_to_bronze` · started 2026-10-11 13:01 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `drift_mode=evolve`, `load_type=all`, `mode=incremental`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 01 · Raw → Bronze
Loads landing batches (JSONL + `_manifest.json`) into the Bronze Delta tables using **explicit StructType
schemas** (no inference). Invalid records go to `so_bronze.quarantine_records`, schema drift is recorded in
`so_ops.schema_drift_events`, and every file processed writes a row to `so_ops.pipeline_execution_logs`.

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: every batch not yet loaded | `backfill`: reload the selected batches |
| `load_type` | `all` | `all`, `full` or `incremental` |
| `batch_ids` | empty | e.g. `incr_20260925T155859Z,full_2025-01-01_2025-04-01` |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `landing_path` | empty | an explicit batch folder, e.g. `/Volumes/workspace/so_raw/landing/incremental/incr_...` |
| `drift_mode` | `evolve` | `evolve` adds new columns (mergeSchema); `rescue` keeps them in `_rescued_data` |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.dropdown("load_type", "all", ["all", "full", "incremental"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("landing_path", "")
dbutils.widgets.dropdown("drift_mode", "evolve", ["evolve", "rescue"])

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.bronze import run_raw_to_bronze

cfg = build_config()
audit = run_raw_to_bronze(
    spark, cfg,
    mode=widget("mode"),
    load_type=widget("load_type"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    landing_path=widget("landing_path") or None,
    fail_on_error=True,
)

Raw-to-Bronze: 5 batch(es) selected, mode=incremental
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
[SKIPPED] Raw-to-Bronze | *                                  | read=0 ins=0 upd=0 del=0 quar=0 | 0.0s | already loaded; use mode=backfill to reload
Raw-to-Bronze summary: {'run_id': 'raw_to_bronze-20261011T130158Z-c1c1a6', 'steps': 5, 'by_status': {'SKIPPED': 5}, 'rows_inserted': 0, 'rows_updated': 0, 'rows_quarantined': 0}


In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
629968ab-8a54-4e97-97a0-50ded613bcde,raw_to_bronze-20261011T130158Z-c1c1a6,raw_to_bronze,Raw-to-Bronze,incremental,full,*,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01,-,2026-10-11T13:02:01.012Z,2026-10-11T13:02:01.012Z,3.7e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-10-11T13:02:01.012Z
651e51a6-f135-4aa6-8af1-db7cdf79840f,raw_to_bronze-20261011T130158Z-c1c1a6,raw_to_bronze,Raw-to-Bronze,incremental,full,*,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01,-,2026-10-11T13:02:02.886Z,2026-10-11T13:02:02.886Z,3.1e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-10-11T13:02:02.886Z
a03cd10f-e607-42e9-b213-64b0a333fba1,raw_to_bronze-20261011T130158Z-c1c1a6,raw_to_bronze,Raw-to-Bronze,incremental,incremental,*,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z,-,2026-10-11T13:02:05.316Z,2026-10-11T13:02:05.316Z,2.8e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-10-11T13:02:05.316Z
d753dee5-ac35-43f2-aceb-9bff82d303cd,raw_to_bronze-20261011T130158Z-c1c1a6,raw_to_bronze,Raw-to-Bronze,incremental,incremental,*,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z,-,2026-10-11T13:02:06.840Z,2026-10-11T13:02:06.840Z,3.3e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-10-11T13:02:06.840Z
c8b25c73-c7ac-4bac-ab15-c2f867d60ae9,raw_to_bronze-20261011T130158Z-c1c1a6,raw_to_bronze,Raw-to-Bronze,incremental,incremental,*,incr_20261007T160033Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T160033Z,-,2026-10-11T13:02:08.293Z,2026-10-11T13:02:08.293Z,2.7e-05,SKIPPED,0,0,0,0,0,already loaded; use mode=backfill to reload,2026-10-11T13:02:08.293Z
